# Squeezing gastruloids

This program was developed for “article” (authors, year) using [CellBasedModels.jl](https://github.com/dsb-lab/CellBasedModels.jl).

The report and the rest of the code can be found on [GitHub](https://github.com/).

## Preamble

### Packages

In [ ]:
println("Running on $(Threads.nthreads()) threads.")

using NBInclude
using DifferentialEquations
using CellBasedModels
using Distributions
using Random
try using GLMakie; Makie.inline!(true) catch; using CairoMakie end
using MathTeXEngine
using Printf
using Dates
using ProgressMeter
using Glob

import CellBasedModels: update!

Makie.update_theme!(fonts = (regular = texfont(), bold = texfont(:bold), italic = texfont(:italic)))

In [ ]:
@nbinclude("preamble/functions-models.ipynb");

### Functions

In [ ]:
color_map = Dict(
    1 => Makie.wong_colors()[1],
    2 => Makie.wong_colors()[3],
    3 => Makie.wong_colors()[2]
)
color_map_alpha = Dict(
    1 => Makie.wong_colors()[1],
    2 => Makie.wong_colors()[3],
    3 => RGBAf(Makie.wong_colors()[2], 0.2f0)
);

In [ ]:
function initialize_growth(parameters; dt)

	com = Community(
		model,
		N = 1,
		dt = dt,
	)

	# Prameters
	for (par, val) in pairs(parameters)
		com[par] = val
	end

	# Initialization
	com.cell_state = 1
	com.x = 0.0
	com.y = 0.0
	com.z = 0.0
	com.vx = 0.0
	com.vy = 0.0
	com.vz = 0.0
	com.t_div = 1

	com.g_on = true
	com.d_on = false

	return com

end;


In [ ]:
function grow_size!(com, save_each, n_cells;
	n_control=100)

	loadToPlatform!(com, preallocateAgents = round(Int, n_cells * 1.1))
	i = 0
	n_total = 0

	while (com.N .< n_cells)
		if com.N > n_total
			println("N > $(n_total)")
			n_total += n_control
		end
		i += 1
		agentStepDE!(com)
		agentStepRule!(com)
		update!(com)
		computeNeighbors!(com)
		if i % save_each == 0
			saveRAM!(com)
		end
	end
	for i in 1:save_each
		i += 1
		agentStepDE!(com)
		update!(com)
	end
	saveRAM!(com)
	bringFromPlatform!(com)
end;


In [ ]:
function initialize_diff!(com;
    g_on=true,
    t_reset=true
    )

    com.g_on = g_on
    com.d_on = true

    if t_reset
        setfield!(com,:t, 0);
    end

    if g_on
        for i in 1:com.N
            t = com.t
            sample_right = 2 * com.tau_div[com.cell_state[i]] * com.sigma_div[] + dt
            for i in 1:com.N
                com.t_div[i] = t + CBMDistributions.uniform(dt, sample_right)
            end
        end
    end
end;

In [ ]:
function initialize_confined_diff!(com; 
    g_on = false,
    t_reset = true,
    r_agg = (maximum(com.x) - minimum(com.x)) * 0.5,
    center = 0.45,
    b0 = 0.7,
    a0 = 0.2
    )

    dt = com.dt
    
    # r_center_2 = (center * r_agg) ^ 2
    # for i in 1:com.N
    #     di0_2 = (com.x[i]^2 + com.y[i]^2 + com.z[i]^2)
    #     if di0_2 < r_center_2
    #         com.cell_state[i] = 3
    #     else com.cell_state[i] = 2
    #     end
    # end

    com.cell_state .= 3
    r_center_2 = (center * r_agg)^2
    central_indices = findall(i -> com.x[i]^2 + com.y[i]^2 + com.z[i]^2 < r_center_2, eachindex(com.x))
    if !isempty(central_indices)
        shuffle!(central_indices)
        n_center = length(central_indices)
        n_b = round(Int, b0 * n_center)
        n_a = round(Int, a0 * n_center)
        com.cell_state[central_indices[1:n_b]] .= 2
        com.cell_state[central_indices[n_b+1:minimum([n_b+1+n_a, n_center])]] .= 1
    end
    
	com.g_on = g_on
    com.d_on = true

    if t_reset
        setfield!(com,:t, 0);
    end

    if g_on
        for i in 1:com.N
            t = com.t
            sample_right = 2 * com.tau_div[com.cell_state[i]] * com.sigma_div[] + dt
            for i in 1:com.N
                com.t_div[i] = t + CBMDistributions.uniform(dt, sample_right)
            end
        end
    end
    
	saveRAM!(com)

end;

In [ ]:
function differentiate!(com, save_each, tf;
    preallocate = 2 * com.N * 2 ^ (tf / mean(com.tau_div))
    )

    message1 = "Possible numerical instabilities. 
            You might have to send argument preallocate = highnumber
            Default is preallocate = 2 * com.N * 2 ^ (tf / mean(com.tau_div))"
    message2 = "Numerical instabilities. 
            You might have to decrease the timestep"

	steps = round(Int64, tf / com.dt)
    step_control = steps / 10
    progress = step_control
    i = 0
    
    if com.g_on[]
        println("Initial N: $(com.N)")
        loadToPlatform!(com, preallocateAgents = round(Int, preallocate))
    else
        loadToPlatform!(com)
    end
    
    for i in 1:steps
		if i > progress
			println("$(round(100 * i / steps))% \t N=$(com.N)")
			progress += step_control
		end
        agentStepDE!(com)
        agentStepRule!(com)
        update!(com)
        computeNeighbors!(com)
        if i % save_each == 0
            saveRAM!(com)
        end
        if all(com.N .> preallocate)
            println(message1)
            break
        end
    end        

	if i % save_each != 0
		saveRAM!(com)
    end
    
	bringFromPlatform!(com)

    println("Final N: $(com.N)")
    
end;


In [ ]:
function stabilize!(com, save_each, tf)

	steps = round(Int64, tf / com.dt)
    step_control = steps / 10
    progress = step_control
	i = 0

	# print("Stabilizing... ")
	prog_bar = Progress(steps, desc="Stabilizing")

	loadToPlatform!(com)

	# while (abs(sum(com.vx)) > 1E-10)
	while (i<steps)
		i += 1
		# if i > progress
		# 	print("$(round(100 * i / steps))% ")
		# 	progress += step_control
		# end
		agentStepDE!(com)
		update!(com)
		if i % save_each == 0
			saveRAM!(com)
		end
		next!(prog_bar)
	end
	if i % save_each != 0
		saveRAM!(com)
	end
	bringFromPlatform!(com)

end;

In [ ]:
function plot_pancake(com, color_map, mstart, mstop;
    boxsize = maximum([
        maximum(com.x) - minimum(com.x),
        maximum(com[mstart].x) - minimum(com[mstart].x)
    ]) / 1.5,
    n = 4,
    showtime = false,
    shownumbers = true,
    filename = "pancake",
    n_views = 3,
    savefig = false
)
    filename = "$(filename)_$(layers)layers_n$(com[mstart].N)"
    views = [
        (perspective = pi/2,  show_z = false, show_x = true, name = "top"),
        (perspective = -pi/2,  show_z = false, show_x = true, name = "bottom"),
        (perspective = 2*pi+pi/32,  show_z = true,  show_x = false, name = "side"),
    ]
    views = views[1:n_views]

    for (j, view) in enumerate(views)
        fig = Figure(size = (n * 640, 600), figure_padding = 40)
        labelsize = 40
        d = getParameter(com, [:x, :y, :z, :r, :cell_state])
        
        for (i, pos) in enumerate(range(start = mstart, length = n, stop = mstop))
            pos = floor(Int, pos)
            if j == 1
                println("Column $i: timestamp $pos")
            end
            t = round(com[pos].t, digits = 2)

            ax = Axis3(
                fig[1, i],
                aspect = :data,
                azimuth = 0,
                elevation = view.perspective,
                xlabel = "",
                ylabel = "",
                zlabel = "",
                xticklabelsize = labelsize,
                yticklabelsize = labelsize,
                zticklabelsize = labelsize,
                titlevisible = showtime,
                titlealign = :center,
                titlegap = 12,
                titlesize = labelsize,
                title = L"t=%$(t)"
            )

            color = [color_map[j] for j in d[:cell_state][pos]]
            meshscatter!(
                ax,
                d[:x][pos],
                d[:y][pos],
                d[:z][pos],
                markersize = d[:r][pos],
                color = color
            )

            xlims!(ax, -boxsize, boxsize)
            ylims!(ax, -boxsize, boxsize)
            zlims!(ax, -boxsize, boxsize)

            if !shownumbers
                ax.xticklabelsize = 0
                ax.yticklabelsize = 0
                ax.zticklabelsize = 0
            end

            ax.zticklabelsvisible = view.show_z
            ax.xticklabelsvisible = view.show_x
            ax.yticklabelsvisible = false
        end
        display(fig)
        if savefig
            save("$(filename)_$(view.name).png", fig)
        end
    end
end;

In [ ]:
function record_pancake(com, color_map, mstart, mstop, layers;
    boxsize = maximum([
        maximum(com.x) - minimum(com.x),
        maximum(com[mstart].x) - minimum(com[mstart].x)
    ]) / 1.5,
    n = (mstop-mstart),
    fps = 30,
    shownumbers = true,
    filename = "pancake",
    n_views = 3,
    size_ratio=1
)
    filename = "$(filename)_$(layers)layers_n$(com[mstart].N)_$(fps)fps.mp4"

    views = [
        (perspective = pi/2,  show_z = false, show_x = true,  name = "top"),
        (perspective = -pi/2,  show_z = false, show_x = true, name = "bottom"),
        (perspective = 2*pi+pi/32,  show_z = true,  show_x = false, name = "side"),
    ]
    views = views[1:n_views]

    d = getParameter(com, [:x, :y, :z, :r, :cell_state])
    frames = floor.(Int, range(start = mstart, length = n, stop = mstop))

    labelsize = 40*size_ratio
    fig = Figure(size = (n_views * 640*size_ratio, 600*size_ratio), figure_padding = 40*size_ratio)

    # Build all axes and scatter plots once
    axes = []
    scatter_plots = []
    pos0 = frames[1]

    for (i, view) in enumerate(views)
        ax = Axis3(
            fig[1, i],
            aspect = :data,
            azimuth = 0,
            elevation = view.perspective,
            xlabel = "", ylabel = "", zlabel = "",
            xticklabelsize = labelsize,
            yticklabelsize = labelsize,
            zticklabelsize = labelsize,
            titlevisible = true,
            titlealign = :center,
            titlegap = 12,
            titlesize = labelsize,
            title = view.name,
        )

        if !shownumbers
            ax.xticklabelsize = 0
            ax.yticklabelsize = 0
            ax.zticklabelsize = 0
        end
        ax.zticklabelsvisible = view.show_z
        ax.xticklabelsvisible = view.show_x
        ax.yticklabelsvisible = false

        xlims!(ax, -boxsize, boxsize)
        ylims!(ax, -boxsize, boxsize)
        zlims!(ax, -boxsize, boxsize)

        color0 = [color_map[cs] for cs in d[:cell_state][pos0]]
        sp = meshscatter!(
            ax,
            d[:x][pos0], d[:y][pos0], d[:z][pos0],
            markersize = d[:r][pos0],
            color = color0,
        )

        push!(axes, ax)
        push!(scatter_plots, sp)
    end

    println("Recording $filename")
    record(fig, filename, frames; framerate = fps) do pos
        # t = round(com[pos].t, digits = 2)
        # println("  frame t=$t")

        color = [color_map[cs] for cs in d[:cell_state][pos]]

        for (ax, sp) in zip(axes, scatter_plots)
            sp[1][] = Point3f.(d[:x][pos], d[:y][pos], d[:z][pos])
            sp.markersize[] = d[:r][pos]
            sp.color[] = color
            # ax.title = L"t=%$(t)"
        end
    end

    println("Saved $filename")
end

In [ ]:
function get_props(com)

	d = getParameter(com, [:t, :cell_state, :N])

	props = Dict()
	for state in 1:3  # 1=A, 2=B, 3=C
		props[state] = [sum(i .== state) for i in d[:cell_state]]
		props[state] = props[state] ./ d[:N]
	end

	return props

end;


In [ ]:
function plot_props(com, color_map, mstart, mstop, props)

    t1 = com[mstart].t
    t2 = com[mstop].t
    
	fig = Figure(resolution = (1000, 800), figure_padding = 25)
    labelsize = 50
		ax = Axis(
        fig[1, 1],
        xlabel = "Signalling time (h)",
        ylabel = "Proportion of cells", 
        xlabelsize = labelsize,
        ylabelsize = labelsize,
        xticklabelsize = labelsize,
        yticklabelsize = labelsize,
        aspect = 1,
        xticks = round.(range(t1, t2, 4), digits=1)
    )
	ylims!(ax, 0, 1)
	xlims!(ax, t1, t2)

	d = getParameter(com, [:t, :cell_state, :N])
	plots = []
	for i in 1:3
		p = lines!(ax, d[:t][mstart:mstop], props[i][mstart:mstop], color = color_map[i], linewidth = 5) # linestyle=:dash
		push!(plots, p)
	end
	labels = [L"state $A$", L"state $B$", L"state $C$"]
    Legend(
        fig[1, 2], 
        plots, labels, 
        labelsize = labelsize,
    )    
	display(fig)

end;


### Model

In [ ]:
model_new = ABM(3,

	# Global parameters
	model = Dict(
		# Mechanics
		:range => Float64,
		:mu => Array{Float64},
		:lambda => Float64,
		:f_rep => Array{Float64},
		:f_att => Array{Float64},
		:alpha_ecm => Float64,
		# Division
		:tau_div => Float64,
		:sigma_div => Float64,
		:olap => Float64,
		:g_on => Bool,
		# Differentiation
		:d_on => Bool,
		:b => Float64,
		:p => Float64,
		:q => Float64,
		:k => Float64,
		# Reference values
		:t0 => Float64,
		:r0 => Float64,
		:f0 => Float64,
		# Confinement
		:confinement => Bool,
		:L => Float64,
		:rep_wall => Float64,
	),


	# Agent parameters
	agent = Dict(
		:t_div => Float64,
		:ni => Int64,
		:cell_state => Int64,
		:r => Float64,
		# Mechanics
		:vx => Float64,
		:vy => Float64,
		:vz => Float64,
		:fx => Float64,
		:fy => Float64,
		:fz => Float64,
		# Protrusions
		:fpx => Float64,
		:fpy => Float64,
		:fpz => Float64,
		:marked => Bool,
		:t_paired => Float64,
		# Differentiation
		:ni_a => Float64,
		:r_ab => Float64,
		:r_bc => Float64,
	),


	# Mechanics
	agentODE = quote

		fx = 0
		fy = 0
		fz = 0
		vsum_x = 0
		vsum_y = 0
		vsum_z = 0
		ni = 0
		state_i = cell_state
		rij = 2 * r

		@loopOverNeighbors it2 begin
			state_j = cell_state[it2]

			dij = CBMMetrics.euclidean(x, x[it2], y, y[it2], z, z[it2])
			mu_ij = mu[state_i, state_j]

			# Cellular forces
			if dij <= mu_ij*rij && dij > 0
				if dij < rij
					f_factor = f_rep[state_i, state_j]
				else
					f_factor = f_att[state_i, state_j]
				end
				inv_dij = 1 / dij
				ratio = rij * inv_dij
				f0_ij = (ratio - 1) * (mu_ij * ratio - 1) * inv_dij
				fx += f_factor * f0_ij * (x - x[it2])
				fy += f_factor * f0_ij * (y - y[it2])
				fz += f_factor * f0_ij * (z - z[it2])
			end

			# Counting neighbours
			if dij < range * rij
				ni += 1
				vsum_x += vx[it2]
				vsum_y += vy[it2]
				vsum_z += vz[it2]
			end
		end

		if confinement
			if z < r
				fz += rep_wall
			end

			diz_top = L - z
			if diz_top < r
				fz -= rep_wall
			end
		end

		# Equations of motion
		ni_relv_min = 5
		if ni < ni_relv_min
			inv = 1 / lambda
			vx = inv * fx
			vy = inv * fy
			vz = inv * fz
		else
			inv = 1 / (lambda * ni)
			vx = inv * (fx + vsum_x*alpha_ecm)
			vy = inv * (fy + vsum_y*alpha_ecm)
			vz = inv * (fz + vsum_z*alpha_ecm)
		end
		dt(x) = vx
		dt(y) = vy
		dt(z) = vz

	end,


	# Growth and differentiation
	agentRule = quote
		# Growth
		if g_on
			if t > t_div
				x_div = CBMDistributions.normal(0, 1)
				y_div = CBMDistributions.normal(0, 1)
				z_div = CBMDistributions.normal(0, 1)
				norm_div = sqrt(x_div^2 + y_div^2 + z_div^2)
				x_div /= norm_div
				y_div /= norm_div
				z_div /= norm_div

				r_sep = r * olap
				@addAgent(
					x = x + r_sep * x_div,
					y = y + r_sep * y_div,
					z = z + r_sep * z_div,
					vx = vx / 2,
					vy = vy / 2,
					vz = vz / 2,
					t_div = t + CBMDistributions.uniform(tau_div[cell_state] * (1 - sigma_div), tau_div * (1 + sigma_div))
				)
				@addAgent(
					x = x - r_sep * x_div,
					y = y - r_sep * y_div,
					z = z - r_sep * z_div,
					vx = vx / 2,
					vy = vy / 2,
					vz = vz / 2,
					t_div = t + CBMDistributions.uniform(tau_div[cell_state] * (1 - sigma_div), tau_div * (1 + sigma_div))
				)
				@removeAgent()
			end
		end

		# Differentiation
		if d_on == true && cell_state != 3
			ni = 0
			ni_a = 0
			@loopOverNeighbors it2 begin
				dij = CBMMetrics.euclidean(x, x[it2], y, y[it2], z, z[it2])
				if dij < range * 2 * r
					ni += 1
					if (cell_state[it2] == 1)
						ni_a += 1
					end
					# if (cell_state[it2] == 1)
					# 	ni_b += 1
					# end
					# if (cell_state[it2] == 1)
					# 	ni_c += 1
					# end
				end
			end

			if ni != 0
				ni_a /= ni
			end

			ran = CBMDistributions.uniform(0, 1)

			if cell_state == 1
				r_ab = p / (1 + k * ni_a)
				if ran < r_ab * dt
					cell_state = 2
				end

			elseif cell_state == 2
				r_bc = q / (1 + k * ni_a)
				if ran < r_bc * dt
					cell_state = 3
				end
			end
		end

	end, 
	
	
	agentAlg = CBMIntegrators.Heun(),
);

### Parameters

In [ ]:
# ORIGINAL MINE

parameters_tfm = Dict(
	:range => 1.2,
	:mu => 2 * [1 1 1; 1 1 1 ; 1 1 1],
	:r => 1,
	:lambda => 1,
	:tau_div => 5 * [1, 1, 1],
	:sigma_div => 0.5,
	:olap => 0.75,
	:p => 0.25,
	:q => 0.125,
	:k => 4.76,
	:f_rep => 2.5 * [1 1 1; 1 1 1 ; 1 1 1],
	:f_att => 		[1 1 1; 1 1 1 ; 1 1 1],
	:t0 => 2,
	:r0 => 5,
	:f0 => 20
);

dt_tfm = 0.002
save_each_tfm = round(Int64, 0.25 / dt_tfm);

In [ ]:
# ORIOLA PAPER

parameters_do = Dict(
	:range => 1,		# changes	
	:mu => 
        [1.55 1.5  1.525
         1.5  1.7  1.45
         2.0  2.0  2.0 ],
	:mu => 2 * [1 1 1; 1 1 1 ; 1 1 1],
	:r => 1,
	:lambda => 1,
	:tau_div => [300, 300, 750],	# changes
	:sigma_div => 0.5,	
	:olap => 0.25,		# changes
	:p => 0.009333,		# changes
	:q => 0.004667,		# changes
	:k => 5,			# changes
	# changes
	:f_rep => 
        [3 3 3. ; 
         3 3 3. ; 
         3 3 5.1],
	:f_att => 
        [6   6   2.4; 
         6   6   1.6; 
         2.4 1.6 4.8],
	:t0 => 2,
	:r0 => 5,
	:f0 => 20
);
dt_do = 0.01;
save_each_do = round(Int64, 0.1 / dt_do);

## Initalization

In [ ]:
model = model_new;

dt = dt_tfm;
save_each = save_each_tfm;
parameters = parameters_tfm

n_cells = 1000;                                  # desired number of cells
layers = 2

Random.seed!(2345)                              # plant seed for reproducibility
com = initialize_growth(parameters; dt = dt);   # initialization. dt can also be set directy

com.confinement = false
com.L = (1.05 * (2*com.r)) * layers
com.rep_wall = 30;

com.alpha_ecm = 0.9 * com.lambda


grow_size!(com, save_each, n_cells)       # grow for a given n_cells
m0 = length(com);

growncom = deepcopy(com);               # backup to go back

# println(formed_correctly(com))          # no sparse cells caused by numerical errors
plot_pancake(com, color_map, 1, m0; savefig=false, filename="growth")

In [ ]:
# println("N = $(com.N)\n")
# println(abs(sum(com.vx)))
# println(abs(sum(com.vy)))
# println(abs(sum(com.vz)))
# println()

# com = deepcopy(growncom)

# setfield!(com,:dt, dt_do)
# save_each = save_each_do;
# for (par, val) in pairs(parameters_do)
# 		com[par] = val
# end

# com.f_att = 0.25 * parameters_do[:f_att]
# stabilize!(com, save_each, 10)

# println("fx: \t", maximum(com.fx), "\t", minimum(com.fx), "\t", mean(abs.(com.fx)))
# println("fy: \t", maximum(com.fy), "\t", minimum(com.fy), "\t", mean(abs.(com.fy)))
# println("fz: \t", maximum(com.fz), "\t", minimum(com.fz), "\t", mean(abs.(com.fz)))

# plot_pancake(com, color_map, m0, length(com); filename = "pancake_change.png")
# # growncom = deepcopy(com)

## Differentiation

In [ ]:
com = deepcopy(growncom)
Random.seed!(2345);

for (par, val) in pairs(parameters_do)
		com[par] = val
end

# com.mu=2
com.f_att = 0.5 * parameters_do[:f_att]
setfield!(com,:dt, dt_do)
save_each = save_each_do

stabilize!(com, save_each, 30)
# initialize_confined_diff!(com, g_on=true; center=0.8, a0=0.15, b0=0.75)
# stabilize!(com, save_each, 20)
initialize_diff!(com)
m1 = length(com);

differentiate!(com, save_each, 400)
m2 = length(com);
evolvedcom = deepcopy(com);

plot_pancake(com, color_map, m1, m2; savefig=false)

### Movie

In [ ]:
plot_pancake(com, color_map_alpha, m1, m2; savefig=false)

In [ ]:
plot_pancake(com, color_map, m1, m2; savefig=true)
record_pancake(com, color_map, m1, m2, layers; fps=180, size_ratio=0.5)

In [ ]:
# pos = m1
# println(count(==(1), com[pos].cell_state) / com[pos].N)
# println(count(==(2), com[pos].cell_state) / com[pos].N)
# println(count(==(3), com[pos].cell_state) / com[pos].N)

## Unconfine

In [ ]:
# com = deepcopy(evolvedcom)
# com.confinement = false
# com.fz=0
# setfield!(com,:dt, 0.005)

# differentiate!(com, save_each, 100)
# m3 = length(com);
# # evolvedcom = deepcopy(com);

# plot_pancake(com, color_map, m2, m3; filename = "pancake_growth.svg", n_views=1)

In [ ]:
# props = get_props(com);      # compute proportion of each state for the saved timestamps
# plot_props(com, color_map, m1, m2, props)